### Capacitated Vehicle Routing Problem (CVRP) - Exact Solver

#### Problem Statement
The Capacitated Vehicle Routing Problem (CVRP) involves a fleet of identical vehicles, each with a fixed carrying capacity, stationed at a central depot. The objective is to design a set of optimal routes to service a set of geographically dispersed customers with known demands, such that:
- Every customer is visited exactly once by exactly one vehicle.
- All vehicle routes originate and terminate at the central depot.
- The total demand of the customers assigned to any single route does not exceed the vehicle's capacity.
- The total routing cost (total distance traveled) is minimized.

In [1]:
#import libraries
import os
import time
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from pathlib import Path
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from vrp_instance_reader import read_vrp_instance

### Mathematical Formulation (2-Index MTZ)

#### Sets and Indices
* $N$: Set of all nodes $\{0, 1, ..., n\}$, where $0$ represents the depot.
* $N_c$: Set of customer nodes $\{1, 2, ..., n\}$.
* $i, j \in N$: Indices for nodes.

#### Parameters
* $c_{ij}$: Travel distance or cost between node $i$ and node $j$.
* $q_i$: Demand of customer $i \in N_c$.
* $Q$: Maximum capacity of each vehicle in the fleet.
* $K$: Maximum number of available vehicles.

#### Decision Variables
* $x_{ij} \in \{0, 1\}$: Binary variable that equals $1$ if a vehicle travels directly from node $i$ to node $j$, and $0$ otherwise.
* $u_i \ge 0$: Continuous variable representing the accumulated load of a vehicle immediately after visiting customer $i \in N_c$.

#### Objective Function
Minimize the total travel distance across all active routes:
$$\text{Minimize} \quad Z = \sum_{i \in N} \sum_{j \in N, j \neq i} c_{ij} x_{ij}$$

#### Constraints

**1. Degree Constraints :**
Every customer must be entered exactly once:
$$\sum_{i \in N, i \neq j} x_{ij} = 1 \quad \forall j \in N_c$$

Every customer must be exited exactly once:
$$\sum_{j \in N, j \neq i} x_{ij} = 1 \quad \forall i \in N_c$$

**2. Fleet Size Constraint:**
The number of vehicles departing from the depot cannot exceed the available fleet $K$:
$$\sum_{j \in N_c} x_{0j} \le K$$
*(Note: Because entering and exiting nodes is strictly balanced, ensuring vehicles leave the depot naturally guarantees they will return).*

**3. Subtour Elimination & Capacity Tracking (MTZ Formulation):**
This constraint serves a dual purpose: it eliminates disconnected subtours among customers and tracks cumulative vehicle load to ensure capacity constraints are not violated. 
$$u_j \ge u_i + q_j - Q(1 - x_{ij}) \quad \forall i, j \in N_c, i \neq j$$

**4. Load Variable Bounds:**
The accumulated load at customer $i$ must be at least its own demand, and cannot exceed the total vehicle capacity $Q$.
$$q_i \le u_i \le Q \quad \forall i \in N_c$$

**5. Binary Domain:**
$$x_{ij} \in \{0, 1\} \quad \forall i, j \in N$$

#### Gurobi Exact Solver (2-Index MTZ)

In [2]:
def gurobi_vrp_exact_2_index(df, K, Q, dist_matrix, time_limit=500):
    
    # --- SETUP DATA ---
    customers = list(range(1, len(df)))  # Customers are indices 1 to N
    depot = 0  # Depot is index 0
    nodes = [depot] + customers  # All locations
    
    # Extract demand for each customer from the dataframe
    demands = {i: float(df.iloc[i]["demand"]) for i in customers}

    # Initialize the Gurobi Model
    mdl = gp.Model("CVRP_Exact_2Index_MTZ")
    
    # Set parameters
    mdl.Params.TimeLimit = time_limit
    mdl.Params.OutputFlag = 0  # 0 to mute console output, 1 to show solver logs

    # --- STEP 1: DECISION VARIABLES ---
    
    # x[i,j] = 1 if ANY vehicle travels directly from node 'i' to node 'j'
    x = mdl.addVars(
        [(i, j) for i in nodes for j in nodes if i != j], 
        vtype=GRB.BINARY, 
        name="x"
    )

    # u[i] = Continuous variable tracking the cumulative load at customer 'i'
    # lb = customer's own demand, ub = capacity Q
    u = mdl.addVars(
        customers, 
        lb=demands, 
        ub=Q, 
        vtype=GRB.CONTINUOUS, 
        name="u"
    )

    # --- STEP 2: OBJECTIVE FUNCTION ---
    mdl.setObjective(
        gp.quicksum(dist_matrix[i][j] * x[i, j] for i in nodes for j in nodes if i != j), 
        GRB.MINIMIZE
    )

    # --- STEP 3: CONSTRAINTS ---

    # Constraint A: Every customer must be entered exactly once
    mdl.addConstrs(
        (gp.quicksum(x[i, j] for i in nodes if i != j) == 1 for j in customers), 
        name="enter"
    )

    # Constraint B: Every customer must be exited exactly once
    mdl.addConstrs(
        (gp.quicksum(x[i, j] for j in nodes if i != j) == 1 for i in customers), 
        name="exit"
    )

    # Constraint C: Fleet Size Limit
    mdl.addConstr(
        gp.quicksum(x[depot, j] for j in customers) <= K, 
        name="max_vehicles"
    )

    # Constraint D: MTZ Subtour Elimination & Capacity Tracking
    mdl.addConstrs(
        (u[j] >= u[i] + demands[j] - Q * (1 - x[i, j]) 
         for i in customers for j in customers if i != j), 
        name="mtz"
    )

    # --- STEP 4: SOLVE THE MODEL ---
    t0 = time.perf_counter()
    mdl.optimize()
    elapsed_secs = time.perf_counter() - t0

    # Check if we have at least one feasible solution
    if mdl.SolCount == 0:
        return [], float("inf"), elapsed_secs, "no_solution", float("inf")

    # Determine status
    if mdl.Status == GRB.OPTIMAL:
        status = "optimal"
    elif mdl.Status == GRB.TIME_LIMIT:
        status = "time_limit_feasible"
    else:
        status = "feasible"

    # Extract gap (Gurobi returns a decimal, multiply by 100 for %)
    gap = mdl.MIPGap * 100.0 if hasattr(mdl, "MIPGap") else 0.0

    # --- STEP 5: EXTRACT THE ROUTES ---
    routes = []
    total_dist = mdl.ObjVal

    # Find start nodes from the depot
    start_nodes = [j for j in customers if x[depot, j].X > 0.5]

    for start_node in start_nodes:
        route = [depot, start_node]
        current = start_node

        while current != depot:
            # Find next node
            next_nodes = [j for j in nodes if j != current and x[current, j].X > 0.5]
            if not next_nodes:
                break
            nxt = next_nodes[0]
            route.append(nxt)
            current = nxt

        routes.append(route)

    return routes, total_dist, elapsed_secs, status, gap

#### Run Manager

In [ ]:
def run_instances(root_dir, time_limit_sec=500):
    folder_path = Path(root_dir)

    # If your txt files are directly in 'Dataset', change the above line to:
    # folder_path = Path(root_dir)

    if not folder_path.exists():
        print(f"Directory not found: {folder_path}")
        return

    files = list(folder_path.glob("*.txt"))
    print(f"Found {len(files)} instances.\n")

    results = []

    for file_path in files:
        instance_name = file_path.name

        try:
            dist_matrix, df, K, Q = read_vrp_instance(str(file_path))
            n_customers = len(df) - 1

            routes, total_dist, elapsed_secs, status, gap = gurobi_vrp_exact_2_index(
                df, K, Q, dist_matrix, time_limit=time_limit_sec
            )

            print(f"  -> {instance_name}: {status.upper()} | Dist: {round(total_dist, 2)} | Time: {round(elapsed_secs, 1)}s | Gap: {round(gap, 2)}%")

            results.append({
                "Instance": instance_name,
                "Customers": n_customers,
                "Vehicles Used": len(routes),
                "Total Distance": round(total_dist, 4) if total_dist != float("inf") else "INFEASIBLE",
                "Time (Seconds)": round(elapsed_secs, 2),
                "Gap (%)": round(gap, 2) if gap != float("inf") else "N/A",
                "Status": status,
                "Routes": str(routes)
            })

        except Exception as e:
            print(f"Error running {instance_name}: {e}")

    # UNINDENTED: This now runs ONCE after the loop finishes
    if results:
        os.makedirs("results", exist_ok=True)
        df_results = pd.DataFrame(results)
        df_results.to_csv("results/gurobi_exact_2index_results.csv", index=False)
        print("\nAll done! Summary saved to: results/gurobi_exact_2index_results.csv")


# UNINDENTED: This is now at the root level of the script, so it will execute!
if __name__ == "__main__":
    # Use a raw string (r"...") so Windows backslashes are read correctly
    ROOT = r"Data\Dataset"
    
    # Call the function 
    run_instances(ROOT, time_limit_sec=1800)

Found 5 instances.

Set parameter Username
Set parameter LicenseID to value 2803432
Academic license - for non-commercial use only - expires 2027-04-04
Set parameter TimeLimit to value 1800
  -> E-n13-k4.txt: OPTIMAL | Dist: 247.0 | Time: 3.2s | Gap: 0.0%
Set parameter TimeLimit to value 1800
  -> E-n23-k3.txt: OPTIMAL | Dist: 568.56 | Time: 74.5s | Gap: 0.0%
Set parameter TimeLimit to value 1800
  -> E-n30-k3.txt: TIME_LIMIT_FEASIBLE | Dist: 542.29 | Time: 1800.2s | Gap: 30.79%
Set parameter TimeLimit to value 1800
  -> E-n51-k5.txt: TIME_LIMIT_FEASIBLE | Dist: 544.68 | Time: 1800.3s | Gap: 19.87%
Set parameter TimeLimit to value 1800
  -> E-n76-k7.txt: TIME_LIMIT_FEASIBLE | Dist: 710.82 | Time: 3008.4s | Gap: 23.3%

All done! Summary saved to: results/gurobi_exact_2index_results.csv
